# Proxy De-risk Track — Qwen-local ICRL + Axis (Colab)

Runs the **Week 1 proxy path** on a single GPU runtime:

1. Generate ~100 ICRL conversations with **local Qwen3-8B** (no Anthropic API)
2. Extract activations → build **proxy axis** (`value_axis_proxy.npy`)
3. Loose gate: L21/L22 AUROC ≥ **0.75** (NOT faithful 0.93)

**Label:** proxy axis for noise feasibility only — not faithful Stage 1.

**Runtime:** A100/L4 strongly recommended. ICRL gen x100 takes several hours on T4.

After this notebook: run Stage 2 SWE-agent pilot (see `stage2/README.md`).

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable GPU: Runtime → Change runtime type → GPU'
print(torch.cuda.get_device_name(0))

In [ ]:
!git clone https://github.com/abdelmagid07/latent_failiure_prediction.git
%cd latent_failiure_prediction/stage1
!pip install -q -e .

In [ ]:
from huggingface_hub import login
login()

In [ ]:
# Pilot: n=10 first, then n=100
N = 10  # change to 100 for full proxy run
OUTPUT = 'data/icrl_proxy_pilot.json' if N <= 10 else 'data/icrl_proxy.json'

!python -m stage1.icrl_gen.generate --n {N} --backend local_qwen \
  --output {OUTPUT} --resume --max-turn-retries 8

In [ ]:
!python -m stage1.pipeline.extract_activations --icrl {OUTPUT} \
  --activations-dir data/activations_proxy --force

In [ ]:
!python -m stage1.pipeline.run_proxy_gate --icrl {OUTPUT} --skip-extract

In [ ]:
from google.colab import files
for p in ['data/value_axis_proxy.npy', 'data/axis_manifest_proxy.json', 'data/auroc_by_layer_proxy.png']:
    files.download(p)